In [1]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from langchain_community.tools import TavilySearchResults
from typing import TypedDict, Annotated
from langgraph.graph import add_messages, StateGraph, END
from dotenv import load_dotenv
from langgraph.prebuilt import ToolNode

load_dotenv()

class AgentState(TypedDict):
    messages: Annotated[list, add_messages]

search_tool = TavilySearchResults(max_results=2)
tools = [search_tool]

llm = ChatGoogleGenerativeAI(model="gemini-2.5-pro")
llm_with_tools = llm.bind_tools(tools=tools)

def model(state: AgentState):
    return {
        "messages": [llm_with_tools.invoke(state["messages"])], 
    }

def tools_router(state: AgentState):
    last_message = state["messages"][-1]

    if(hasattr(last_message, "tool_calls") and len(last_message.tool_calls) > 0):
        return "tool_node"
    else: 
        return END
    

tool_node = ToolNode(tools=tools)

graph = StateGraph(AgentState)

graph.add_node("model", model)
graph.add_node("tool_node", tool_node)
graph.set_entry_point("model")

graph.add_conditional_edges("model", tools_router)
graph.add_edge("tool_node", "model")

app = graph.compile()

C:\Users\user\AppData\Local\Temp\ipykernel_7060\4196788894.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import TavilySearchResults
C:\Users\user\AppData\Local\Temp\ipykernel_7060\4196788894.py:14: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  search_tool = TavilySearchResults(max_results=2)


In [6]:
input = {
    "messages": ["What's the current weather in Bangalore?"]
}

events=app.stream(input=input, stream_mode="updates")

for event in events:
    print(event)

{'model': {'messages': [AIMessage(content='', additional_kwargs={'function_call': {'name': 'tavily_search_results_json', 'arguments': '{"query": "current weather in Bangalore"}'}, '__gemini_function_call_thought_signatures__': {'d7ced338-0ec3-48d5-bb0d-07b902ac35cd': 'CpkEARFNMg+ZJ/QDz1ns3nnJ2SSSDRzAY4uoXQHm3hwrI3e0RQK5W3n4linARdqLYmF3b5IGIY0D6B6e6zT+MDZytQXSRibFw5Gjsv2fdKt4gz3jFxm+vZN8soIGFWNuQS4Gs2tyxwg6EMk7DJtZ5gzih6+caoLCmqsBywyh3SKGnpOl0XxapbO1+802kgtDOBbp5cj8O8OGOjFvn7O7sbe+08AXh9vBVx6q01MMzZoPwaLkrJYhoJBB6plrhP9p8ylDPMPIZ685HK8LSY//BC37JNNFmp99yGHvVjkkkSgo6u0aq2YZco/hZjyhIbtp/q7dyHeIzI38R8JaB3rS39QOKVTYwWrb1JJi4YDUPif6P4GqDlxDNmOZ2y3B7TiGv6UL56JApOJcku39DHC654JFAxUUGEL0WNDxeamCH/cssN8NAPRWAU80ccU+l1q7TZ+mvJ6V2SVvmCvmghrplAaPgOjue+C0cXu/69UxZ8TBJ4c0mUFFIr3ztPbQiacBOX7yxkaab6V/DT8DX0Oc9Rh8a/cGvm6H4jl+2ZuO+8qPPmGvl6seEKYwxQGuGH+81QcQciMmFga8f3ZcL6qMKgwI/UW7My8nnr5ncV5Brwvllg0TO+0oSCG5TYqZDwtXMyE9VdssiBkEXu3N/YfQrpI4WrH4Q3LuYobZ/Sx3PyTCI6fPWWw+jLQ9rKn+TwbVzHyiSatpT3m8Zdsv'}}, respon

In [20]:
input = {
    "messages": ["when is the portugal vs Spain match airing in Malaysia?"]
}

events = app.astream_events(input=input, version="v2")

async for event in events: 
    print(event,flush=True)

{'event': 'on_chain_start', 'data': {'input': {'messages': ['when is the portugal vs Spain match airing in Malaysia?']}}, 'name': 'LangGraph', 'tags': [], 'run_id': '019f384c-cedc-7791-860a-0aa7722883bb', 'metadata': {'ls_integration': 'langgraph'}, 'parent_ids': []}
{'event': 'on_chain_start', 'data': {'input': {'messages': [HumanMessage(content='when is the portugal vs Spain match airing in Malaysia?', additional_kwargs={}, response_metadata={}, id='db6a56a0-59e3-4c0b-b296-f9e12369c7b6')]}}, 'name': 'model', 'tags': ['graph:step:1'], 'run_id': '019f384c-cee9-79d0-a418-9bc2d9856728', 'metadata': {'ls_integration': 'langgraph', 'langgraph_step': 1, 'langgraph_node': 'model', 'langgraph_triggers': ('branch:to:model',), 'langgraph_path': ('__pregel_pull', 'model'), 'langgraph_checkpoint_ns': 'model:d26d4e22-21ad-053d-b444-9f7dd2bd1d88'}, 'parent_ids': ['019f384c-cedc-7791-860a-0aa7722883bb']}
{'event': 'on_chat_model_start', 'data': {'input': {'messages': [[HumanMessage(content='when is 